# 範例 2：Singularity 容器 + HUMAnN3 互動測試

對應教學文件：[03_Singularity_HUMAnN3_教學.md](03_Singularity_HUMAnN3_教學.md)

這份 notebook 只做「**互動測試**」——確認容器裝得對、指令跑得動、資料庫下載得下來。
真正跑分析（尤其是正式研究等級的完整資料庫）請用同資料夾的
[run_humann3_pipeline.sh](run_humann3_pipeline.sh) + [slurm_humann3_pipeline.sh](slurm_humann3_pipeline.sh)
透過 SLURM 送出，不要在 notebook/login node 裡跑重運算。

> Kernel 不需要特別設定，任何能跑 `%%bash` 的 Python kernel 都可以，這個範例完全不依賴 conda 環境
> （所有依賴都封裝在 `.sif` 容器裡），詳見教學文件第 0 節。


## Step 1：確認 Singularity 與容器 image 都在

In [ ]:
%%bash
singularity --version
echo "---"
ls -la /work/c00cjz00/notebook/class/03_singularity_humann3/containers/humann_latest.sif


## Step 2：把 Docker Hub 上的 image 轉成 Singularity `.sif`（第一次才需要）

已經幫大家轉好放在 `containers/humann_latest.sif` 了，這裡列出當初的指令供參考，
**不需要重新執行**（除非你要更新到新版本，或自己在別的機器上重做一次）：

```bash
mkdir -p containers
singularity pull --force containers/humann_latest.sif docker://biobakery/humann:latest
```


## Step 3：確認容器內的 HUMAnN3 可以正常執行

In [ ]:
%%bash
SIF=/work/c00cjz00/notebook/class/03_singularity_humann3/containers/humann_latest.sif

echo "=== humann --version ==="
singularity exec --bind /work/c00cjz00:/work/c00cjz00 "$SIF" humann --version

echo ""
echo "=== 可用的資料庫選項（找 DEMO 版本用）==="
singularity exec --bind /work/c00cjz00:/work/c00cjz00 "$SIF" humann_databases --available


## Step 4：下載官方 DEMO 資料庫（chocophlan + uniref）

正式版資料庫太大（ChocoPhlAn 數 GB、UniRef 動輒 20GB+），課堂上改用官方提供的 `DEMO` 迷你版，
兩個都是幾秒鐘就能下載完的等級。


In [ ]:
%%bash
SIF=/work/c00cjz00/notebook/class/03_singularity_humann3/containers/humann_latest.sif
DB=/work/c00cjz00/notebook/class/03_singularity_humann3/db
mkdir -p "$DB"

singularity exec --bind /work/c00cjz00:/work/c00cjz00 "$SIF" \
  humann_databases --download chocophlan DEMO "$DB" --update-config no

singularity exec --bind /work/c00cjz00:/work/c00cjz00 "$SIF" \
  humann_databases --download uniref DEMO_diamond "$DB" --update-config no

echo "=== 資料庫大小 ==="
du -sh "$DB"/*


## Step 5：跑一次完整 HUMAnN3 分析（用容器內建的官方示範 FASTQ）

容器裡本來就內建了 HUMAnN3 官方自己出的示範資料：
`/usr/local/lib/python3.6/dist-packages/humann/tests/data/demo.fastq`，不用自己準備輸入檔案。

**關鍵參數 `--bypass-prescreen`**：HUMAnN3 預設第一步會用 MetaPhlAn 做物種篩選，
但 MetaPhlAn 的物種標記基因資料庫**完整版接近 40GB**，完全不適合課堂 demo。
加上這個參數可以跳過篩選，直接對（已經是小型 DEMO 版的）ChocoPhlAn 全庫比對，
详细踩坑過程見教學文件第 5.1 節。


In [ ]:
%%bash
SIF=/work/c00cjz00/notebook/class/03_singularity_humann3/containers/humann_latest.sif
DB=/work/c00cjz00/notebook/class/03_singularity_humann3/db
OUT=/work/c00cjz00/notebook/class/03_singularity_humann3/demo_out
mkdir -p "$OUT"

time singularity exec --bind /work/c00cjz00:/work/c00cjz00 "$SIF" \
  humann --input /usr/local/lib/python3.6/dist-packages/humann/tests/data/demo.fastq \
  --output "$OUT" \
  --nucleotide-database "$DB/chocophlan" \
  --protein-database "$DB/uniref" \
  --bypass-prescreen \
  --threads 4

echo ""
echo "=== 輸出檔案 ==="
ls -la "$OUT"


## Step 6：看結果

In [ ]:
import pandas as pd

genefamilies = pd.read_csv(
    "/work/c00cjz00/notebook/class/03_singularity_humann3/demo_out/demo_genefamilies.tsv",
    sep="\t"
)
genefamilies.head(15)


In [ ]:
pathabundance = pd.read_csv(
    "/work/c00cjz00/notebook/class/03_singularity_humann3/demo_out/demo_pathabundance.tsv",
    sep="\t"
)
pathabundance.head(15)


---

## 練習題

1. `demo_genefamilies.tsv` 裡的基因家族名稱格式是 `UniRef90_xxx|g__屬.s__種`，
   找出這份 demo 結果裡出現了哪幾個物種？跟教學文件第 5.1 節提到的實測結果對得上嗎？
2. 如果拿掉 `--bypass-prescreen`，第一步 MetaPhlAn 會嘗試下載多大的資料庫？
   為什麼正式研究分析通常還是需要這一步，不能永遠依賴 `--bypass-prescreen`？
3. 打開 [run_humann3_pipeline.sh](run_humann3_pipeline.sh)，
   對照這裡在 notebook 裡手動下的每一條指令，找出腳本裡對應的段落。
4. 修改 [slurm_humann3_pipeline.sh](slurm_humann3_pipeline.sh) 的 partition，
   實際送出一次 SLURM job，用 `squeue`/`sacct` 追蹤到完成。
